# Mở SSH trên Colab để nối với VS Code / code-server

Máy nhà (code-server) **không dùng được extension Remote-SSH** (chỉ VS Code desktop mới có).
Nên có 2 mức:
- **Mức 1 (khuyên dùng):** khỏi cần notebook này — sửa code trong code-server → `git push` → Colab `git pull` (xem `notebooks/train_colab.ipynb`).
- **Mức 2 (nâng cao):** chạy notebook này để Colab mở cổng SSH (qua ngrok), rồi từ **terminal của code-server** gõ lệnh `ssh` in ở cuối notebook — gõ lệnh chạy ngay trên máy Colab, còn file code vẫn sửa trong editor của code-server.

**Cần 1 token ngrok miễn phí:** đăng ký tại https://dashboard.ngrok.com → copy authtoken.

In [ ]:
# Bước 1: đặt mật khẩu root cho SSH trên máy Colab
import getpass
PWD = getpass.getpass('Chọn mật khẩu SSH cho máy Colab: ')
!echo root:{PWD} | chpasswd
!apt-get install -y -qq openssh-server >/dev/null 2>&1 || true
!mkdir -p /var/run/sshd
!(pgrep sshd >/dev/null || /usr/sbin/sshd)
print('sshd OK')

In [ ]:
# Bước 2: cài ngrok + authtoken (dán token của bạn vào)
NGROK_TOKEN = "DÁN_AUTHTOKEN_CỦA_BẠN_VÀO_ĐÂY"
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xzf ngrok-v3-stable-linux-amd64.tgz
!./ngrok authtoken {NGROK_TOKEN}
print('ngrok OK')

In [ ]:
# Bước 3: mở tunnel TCP tới cổng 22 và in lệnh ssh
import json, time, urllib.request
get_ipython().system_raw('./ngrok tcp 22 --log stdout --log-format jsonraw &')
cmd = None
for _ in range(30):
    time.sleep(1)
    try:
        cfg = json.load(urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels'))
        addr = cfg['tunnels'][0]['public_url']           # ví dụ: tcp://0.tcp.ap.ngrok.io:12345
        host, port = addr.replace('tcp://', '').rsplit(':', 1)
        cmd = f'ssh root@{host} -p {port}'
        break
    except Exception:
        pass
print()
print('=' * 60)
if cmd:
    print('Mở terminal của code-server rồi gõ:  ' + cmd)
    print('(mật khẩu là mật khẩu vừa đặt ở Bước 1)')
else:
    print('CHƯA CÓ TUNNEL — kiểm tra lại NGROK_TOKEN ở Bước 2.')
print('=' * 60)

## Bên code-server (máy nhà)

```bash
# gõ trong terminal của code-server:
ssh root@0.tcp.ap.ngrok.io -p <PORT>     # lấy từ output trên
```

Muốn gọn thì lưu config 1 lần (`~/.ssh/config`):
```
Host colab
    HostName 0.tcp.ap.ngrok.io
    Port <PORT>
    User root
```
→ sau đó chỉ cần `ssh colab`. Lưu ý: **mỗi lần chạy lại notebook là PORT đổi**, phải sửa lại Port.

Nếu sau này dùng **VS Code desktop** (không phải code-server): cài extension *Remote - SSH*, config y như trên → `Connect to Host... → colab` là mở được nguyên cửa sổ VS Code chạy trên máy Colab.